In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
import keras
import torch


# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


### Residual connections

In [3]:
# The target block changing the number of output filters
inputs = keras.Input(shape=(32, 32, 3), name="Input layer")
x = keras.layers.Conv2D(32, 3, activation="relu", name="Conv2D 1")(inputs)
residual = x
x = keras.layers.Conv2D(64, 3, activation="relu", padding="same", name="Conv2D 2")(x)
residual = keras.layers.Conv2D(64, 1, name="Residual 1 [From Conv2D 1]")(residual)
x = keras.layers.add([x, residual], name="Conv2D 2 and Residual")


outputs =keras.layers.Dense(1, activation="sigmoid", name="Output layer")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="Residual Model A")
model.summary()

Model: "Residual Model A"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2D 1 (Conv2D)   │ (None, 30, 30,    │        896 │ Input layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2D 2 (Conv2D)   │ (None, 30, 30,    │     18,496 │ Conv2D 1[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Residual 1 [From    │ (None, 30, 30,    │      2,112 │ Conv2D 1[0][0]    │
│ Conv2D 1] (Conv2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2D 2 and        │ (None, 30, 30,    │          0 │ Conv2D 2[0][0],   │
│ Residual (Add)      │ 64)               │            │ Residual 1 [From  │
│                     │                   │            │ Conv2D 1][0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output layer        │ (None, 30, 30, 1) │         65 │ Conv2D 2 and      │
│ (Dense)             │                   │            │ Residual[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# The target block including a max pooling layer
inputs = keras.Input(shape=(32, 32, 3), name="Input layer")
x = keras.layers.Conv2D(32, 3, activation="relu", name="Conv2D 1")(inputs)
residual = x
x = keras.layers.Conv2D(64, 3, activation="relu", padding="same", name="Conv2D 2")(x)
x = keras.layers.MaxPooling2D(2, padding="same", name="Max Pool 1")(x)
residual = keras.layers.Conv2D(64, 1, strides=2, name="Residual 1 [From Conv2D 1]")(residual)
x = keras.layers.add([x, residual], name="Max Pool 1 and Residual 1")

outputs =keras.layers.Dense(1, activation="sigmoid", name="Output layer")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="Residual Model B")
model.summary()

Model: "Residual Model B"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2D 1 (Conv2D)   │ (None, 30, 30,    │        896 │ Input layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv2D 2 (Conv2D)   │ (None, 30, 30,    │     18,496 │ Conv2D 1[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Max Pool 1          │ (None, 15, 15,    │          0 │ Conv2D 2[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Residual 1 [From    │ (None, 15, 15,    │      2,112 │ Conv2D 1[0][0]    │
│ Conv2D 1] (Conv2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Max Pool 1 and      │ (None, 15, 15,    │          0 │ Max Pool 1[0][0], │
│ Residual 1 (Add)    │ 64)               │            │ Residual 1 [From  │
│                     │                   │            │ Conv2D 1][0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output layer        │ (None, 15, 15, 1) │         65 │ Max Pool 1 and    │
│ (Dense)             │                   │            │ Residual 1[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,569 (84.25 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# generic Kind
inputs = keras.Input(shape=(32, 32, 3), name="Input layer")
x = keras.layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False, i=0):
    residual = x
    x = keras.layers.Conv2D(filters, 3, activation="relu", padding="same", name=f"{i} Conv2D 1")(x)
    x = keras.layers.Conv2D(filters, 3, activation="relu", padding="same", name=f"{i} Conv2D 2")(x)
    if pooling:
        x = keras.layers.MaxPooling2D(2, padding="same", name=f"{i} Maxpool 1")(x)
        residual = keras.layers.Conv2D(filters, 1, strides=2, name=f"{i} Residual [From First]")(residual)
    elif filters != residual.shape[-1]:
        residual = keras.layers.Conv2D(filters, 1, name=f"{i} Residual [From First]")(residual)
    x = keras.layers.add([x, residual], name=f"{i} X + Residual")
    return x


x = residual_block(x, filters=32, pooling=True, i=1)
x = residual_block(x, filters=64, pooling=True, i=2)
x = residual_block(x, filters=128, pooling=False, i=3)
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(1, activation="sigmoid", name="Output Layer")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="Generic Residual Model")
model.summary()

Model: "Generic Residual Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 32, 32, 3) │          0 │ Input layer[0][0] │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1 Conv2D 1 (Conv2D) │ (None, 32, 32,    │        896 │ rescaling[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1 Conv2D 2 (Conv2D) │ (None, 32, 32,    │      9,248 │ 1 Conv2D 1[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1 Maxpool 1         │ (None, 16, 16,    │          0 │ 1 Conv2D 2[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1 Residual [From    │ (None, 16, 16,    │        128 │ rescaling[0][0]   │
│ First] (Conv2D)     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 1 X + Residual      │ (None, 16, 16,    │          0 │ 1 Maxpool         │
│ (Add)               │ 32)               │            │ 1[0][0], 1        │
│                     │                   │            │ Residual [From    │
│                     │                   │            │ First][0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 2 Conv2D 1 (Conv2D) │ (None, 16, 16,    │     18,496 │ 1 X +             │
│                     │ 64)               │            │ Residual[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 2 Conv2D 2 (Conv2D) │ (None, 16, 16,    │     36,928 │ 2 Conv2D 1[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 2 Maxpool 1         │ (None, 8, 8, 64)  │          0 │ 2 Conv2D 2[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 2 Residual [From    │ (None, 8, 8, 64)  │      2,112 │ 1 X +             │
│ First] (Conv2D)     │                   │            │ Residual[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 2 X + Residual      │ (None, 8, 8, 64)  │          0 │ 2 Maxpool         │
│ (Add)               │                   │            │ 1[0][0], 2        │
│                     │                   │            │ Residual [From    │
│                     │                   │            │ First][0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 3 Conv2D 1 (Conv2D) │ (None, 8, 8, 128) │     73,856 │ 2 X +             │
│                     │                   │            │ Residual[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 3 Conv2D 2 (Conv2D) │ (None, 8, 8, 128) │    147,584 │ 3 Conv2D 1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 3 Residual [From    │ (None, 8, 8, 128) │      8,320 │ 2 X +             │
│ First] (Conv2D)     │                   │            │ Residual[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ 3 X + Residual      │ (None, 8, 8, 128) │          0 │ 3 Conv2D 2[0][0]

 Total params: 297,697 (1.14 MB)

 Trainable params: 297,697 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

### Batch Normalization

In [6]:
# batch Normalization after conv2d
def batch_normalize(x):
    x = keras.layers.Conv2D(32, 3, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)

In [7]:
# batch Normalization after conv2d before activation
def batch_normalize(x):
    x = keras.layers.Conv2D(32, 3, use_bias=False)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

### Depthwise separable convolutions